# Step 1 — Collect CellRanger QC Metrics

## The problem this step solves

You are about to spend hours running a multi-step cleaning and integration pipeline across 42 samples. Three of those samples will fail — not obviously, but in ways that would silently corrupt your atlas if you included them. How do you find them before you waste the compute?

CellRanger is the upstream tool (run on a compute cluster, not in this notebook) that converts raw FASTQ sequencing reads into a count matrix: a table of cells × genes where each value is the number of times that gene's mRNA was detected in that cell. As part of that conversion, it writes a one-row quality summary per sample — `metrics_summary.csv`. This notebook collects those summaries across all samples into a single table so you can see, side by side, whether any samples look anomalous before committing to the full pipeline.

## Why this matters for the Yang et al. paper

The paper profiled **42 samples** across three tissues (scWAT, vWAT, skeletal muscle) and four experimental conditions. **Three libraries failed** — D19-5431, D19-5443, and D19-5462 — and were excluded. The remaining 39 samples yielded the 204,883 cells in the final atlas. Identifying those three failures at this step, before running SoupX, DoubletFinder, and integration on them, saved substantial compute time and prevented corrupted data from entering the atlas.

This step also informs the per-sample QC thresholds used in Step 3. A sample with median 3,000 genes per cell should not use the same lower-bound filter as one with median 800 — the QC table makes those differences visible.

## What goes wrong if you skip this

A failed library with <500 detected cells enters the pipeline, passes per-sample filters (its cells look "normal" individually), and gets integrated into the atlas. In the embedding, it contributes a sparse, noisy cluster that may be annotated as a rare cell type. The paper's conclusion about MSC responses to exercise could include artifacts from a bad library.

> **ML analogy:** This is dataset auditing before training. You would never start a model run without first checking label quality, class balance, and whether any data batches are corrupted. This step is the equivalent for scRNA-seq — a per-sample "health check" table before the expensive pipeline runs.

## What this notebook does

This notebook aggregates the per-sample `metrics_summary.csv` files into a single table — `cellranger_metrics.txt` — that gives a side-by-side quality overview of all samples before any downstream filtering.

**Input:** a tab-delimited manifest with three columns:
- `folder_path` — path to the CellRanger `outs/` directory for a given library
- `library_ID` — unique identifier for the sequencing library (one per biological sample in this dataset)
- `sample_ID` — biological sample identifier; a sample may aggregate multiple libraries

**Output:** `cellranger_metrics.txt` — a tab-delimited table with all per-library metrics concatenated, plus a `library` column identifying the source library.

In [1]:
import pandas as pd
from pathlib import Path

## Configuration

Set the path to the manifest file and the output directory before running. The manifest must be tab-delimited with a header row containing `folder_path`, `library_ID`, and `sample_ID`.

In [2]:
# Path to the tab-delimited manifest
cellranger_input_file = "cellranger_manifest.txt"

# Directory where cellranger_metrics.txt will be written
target_folder = "."

## Load the manifest

The manifest maps each library to its CellRanger output directory. Each row represents one library (one sequencing run). Multiple libraries can share a `sample_ID` if a biological sample was split across sequencing runs, but in this dataset each sample has exactly one library.

In [4]:
import pandas as pd

manifest = pd.read_csv(cellranger_input_file, sep="\t")
print(f"Loaded manifest with {len(manifest)} samples")
manifest.head()

FileNotFoundError: [Errno 2] No such file or directory: 'cellranger_manifest.txt'

## Collect metrics from each library

For each row in the manifest, we read the `metrics_summary.csv` from the corresponding CellRanger `outs/` directory, tag it with the `library_ID`, and concatenate all libraries into one table.

CellRanger formats numeric columns with commas as thousands separators (e.g. `"1,234"`) and appends `%` to percentage columns (e.g. `"98.5%"`). We strip those so all values are numeric and can be used directly in comparisons and downstream filtering.

In [ ]:
frames = []

for _, row in manifest.iterrows():
    metrics_path = Path(row["folder_path"]) / "metrics_summary.csv"
    print(f"Reading: {row['library_ID']} — {metrics_path}")

    df = pd.read_csv(metrics_path)

    # CellRanger formats numbers with commas and percentages with '%';
    # strip those so columns are numeric.
    df = df.apply(
        lambda col: pd.to_numeric(
            col.astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False),
            errors="ignore",
        )
    )

    df["library"] = row["library_ID"]
    frames.append(df)

metrics = pd.concat(frames, ignore_index=True)
print(f"\nCombined table: {metrics.shape[0]} rows × {metrics.shape[1]} columns")
metrics.head()

## Key metrics and their biological meaning

Before writing the output, it is worth reviewing what each CellRanger metric actually tells us. The `metrics_summary.csv` columns vary slightly between CellRanger versions, but the most important ones are:

### Cell detection
| Metric | What it means |
|--------|--------------|
| `Estimated Number of Cells` | CellRanger's call on how many barcodes correspond to real cells (using the "knee" inflection point in the UMI barcode rank plot). Very low counts may indicate failed dissociation or poor cell viability. Very high counts may indicate doublets or ambient RNA contamination. |
| `Median Genes per Cell` | The median number of unique genes detected per cell. Low values (< ~500 for most tissues) suggest low-quality cells or shallow sequencing. High values are generally good, though highly variable values across samples can indicate batch effects. |
| `Median UMI Counts per Cell` | Total transcript molecules captured per cell (UMIs = unique molecular identifiers deduplicate PCR copies). Correlates with sequencing depth and library quality. |

### Sequencing quality
| Metric | What it means |
|--------|--------------|
| `Sequencing Saturation` | The fraction of reads that are duplicates of already-observed UMIs. Values below ~50% mean the library is under-sequenced and more reads would discover new transcripts. Values above ~80% mean sequencing is deep enough that additional reads yield diminishing returns. |
| `Q30 Bases in RNA Read` | Percentage of bases in RNA reads with Phred quality score ≥ 30 (i.e., < 0.1% base call error). Below ~65% suggests sequencing run quality issues. |

### Mapping and alignment
| Metric | What it means |
|--------|--------------|
| `Reads Mapped to Genome` | Percentage of reads that aligned to the reference genome. Low mapping rates (< 70%) can indicate contamination, wrong reference, or poor RNA quality. |
| `Reads Mapped Confidently to Transcriptome` | The subset of mapped reads that align unambiguously to annotated transcripts and are used for UMI counting. This is the most relevant number for expression quantification. |
| `Reads Mapped Antisense to Gene` | A high antisense mapping rate can indicate issues with library preparation (e.g., template switching) or problems with the genome annotation. |

### Why compare across samples?
In a multi-sample study like this one (multiple tissues, diet and exercise conditions), large differences in these metrics between samples can confound biological comparisons. For example, if scWAT samples have twice the median UMI count of SkM samples purely due to library prep differences, apparent transcriptional differences between tissues may partly reflect technical noise. Reviewing this table early allows you to:

1. Exclude samples that failed QC before running the expensive processing pipeline.
2. Decide whether to apply depth normalization or other batch correction strategies.
3. Set per-sample (rather than global) filtering thresholds in Step 3, since a "low quality" cutoff appropriate for one tissue may be too aggressive or too lenient for another.

A summary of the key per-sample metrics helps identify any outlier libraries before committing to the full processing pipeline.

In [ ]:
# Show the columns present — useful since column names differ slightly between CellRanger versions
print("Columns:", metrics.columns.tolist())

# Summary statistics across samples.
# Look for:
#   - Low "Estimated Number of Cells" (< 1,000) → possible dissociation failure
#   - Low "Median Genes per Cell" (< 500) → shallow sequencing or poor quality cells
#   - Low "Sequencing Saturation" (< 50%) → library is under-sequenced
#   - Low "Reads Mapped Confidently to Transcriptome" (< 50%) → alignment or contamination issue
#   - High variance across samples → may require per-sample QC thresholds in Step 3
metrics.describe()

## Write output

The combined metrics table is written to `cellranger_metrics.txt` as a tab-delimited file.

This file serves two purposes downstream:
1. **Sample exclusion** — any sample that falls clearly below acceptable thresholds on the metrics above can be dropped here, before it enters the per-sample processing pipeline (Step 3). It is far cheaper to exclude a sample at this stage than after hours of processing.
2. **Threshold setting** — the per-sample median genes and UMI counts inform the nFeature/nCount filtering thresholds in Step 3. Rather than applying a single global cutoff, you can use this table to set thresholds relative to each sample's expected range, reducing the risk of over-filtering high-quality cells in deep samples or under-filtering low-quality cells in shallow ones.

In [ ]:
output_path = Path(target_folder) / "cellranger_metrics.txt"
metrics.to_csv(output_path, sep="\t", index=False)
print(f"Written: {output_path}")